In [1]:
import os
from pathlib import Path

import pandas as pd

from shadowroutes.config import settings

src_path = Path('../')
os.chdir(src_path)

## Parameters

In [23]:
# Mining concessions (Censo Nacional de Seguridad Pública Estatal 2025 - INEGI) - obtained in November 2025
pnl_pol_path = Path(settings.DATA_ROOT / 'cnspf-e_inegi' / 'raw' / 'pers_func_segpub_cngmd2023_csv' / 'conjunto_de_datos' / 'm3s1p14_cngmd2023.csv')
pnl_fed_path = Path(settings.DATA_ROOT / 'cnspf-e_inegi' / 'raw' / 'pers_func_segpub_cngmd2023_csv' / 'conjunto_de_datos' / 'm3s1p15_cngmd2023.csv')

# Population estimates (mid-year) from CONAPO - obtained in October 2025
data_path_pop_m = Path(settings.DATA_ROOT / 'conapo' / 'raw' / 'pobproy_quinq1.csv')

# Output path
out_protection_panel = Path(settings.DATA_ROOT / 'processed' / 'protection_index_panel.csv')

## Data

In [3]:
# Loads data related to personnel in police corporations
df_num_pol = pd.read_csv(pnl_pol_path, encoding='latin1')
print(f'File read: {print(pnl_pol_path.name)}')

print(df_num_pol.shape)
df_num_pol.head()

m3s1p14_cngmd2023.csv
File read: None
(9900, 5)


,ubicageo_f,funcsepu_a,sexostt,sexos1,sexos2
0,1001,1,0,0,0
1,1001,2,3,0,3
2,1001,3,23,14,9
3,1001,4,NSS,NSS,NSS
4,1002,1,0,0,0


In [4]:
# Loads data related to personnel in federal law enforcement institutions
df_num_fed = pd.read_csv(pnl_fed_path, encoding='latin1')
print(f'File read: {print(pnl_fed_path.name)}')

print(df_num_fed.shape)
df_num_fed.head()

m3s1p15_cngmd2023.csv
File read: None
(7425, 9)


,ubicageo_f,tipoinst_b,rnsnni1,perioatt,perioa1,perioa2,perioa3,perioa4,perioa5
0,1001,1,2,NaN,NaN,NaN,NaN,NaN,NaN
1,1001,2,2,NaN,NaN,NaN,NaN,NaN,NaN
2,1001,3,2,NaN,NaN,NaN,NaN,NaN,NaN
3,1002,1,2,NaN,NaN,NaN,NaN,NaN,NaN
4,1002,2,2,NaN,NaN,NaN,NaN,NaN,NaN


### Preprocessing

#### Police personnel per municipality

In [5]:
df_num_pol["sexostt"] = pd.to_numeric(df_num_pol["sexostt"], errors="coerce")
df_num_pol["sexostt"] = df_num_pol["sexostt"].fillna(0)

df_pol_mun = df_num_pol.groupby(['ubicageo_f'], as_index=False)['sexostt'].sum()
df_pol_mun.rename(columns={'ubicageo_f': 'muni_id', 'sexostt': 'count_pol_mun'}, inplace=True)

print(df_pol_mun.shape)
df_pol_mun.head()

(2475, 2)


,muni_id,count_pol_mun
0,1001,26.0
1,1002,59.0
2,1003,112.0
3,1004,30.0
4,1005,167.0


#### Federal presence of law enforcement per municipality

In [6]:
df_num_fed.rnsnni1.value_counts()

rnsnni1
2     6783
0      270
9      193
1      176
98       3
Name: count, dtype: int64

In [7]:
df_num_fed["federal_presence"] = df_num_fed["rnsnni1"].replace({
    1: 1,    # Sí
    2: 0,    # No
    0: 0,    # No aplica
    9: 0,    # No identificado
    96: 0,
    97: 0,
    98: 0
})

# If any federal force was present, mark presence=1
df_fed_mun = df_num_fed.groupby("ubicageo_f", as_index=False)["federal_presence"].max()
df_fed_mun.rename(columns={"ubicageo_f": "muni_id"}, inplace=True)

print(df_fed_mun.shape)
df_fed_mun.head()

(2475, 2)


,muni_id,federal_presence
0,1001,0
1,1002,0
2,1003,0
3,1004,0
4,1005,0


In [19]:
df_fed_mun.federal_presence.value_counts()

federal_presence
0    2358
1     117
Name: count, dtype: int64

#### Population estimates

In [9]:
data_pop_m = pd.read_csv(data_path_pop_m, encoding='utf-8')
print(f'File read: {print(data_path_pop_m.name)}')

print(data_pop_m.shape)
data_pop_m.head()

pobproy_quinq1.csv
File read: None
(252450, 27)


,CLAVE,CLAVE_ENT,NOM_ENT,NOM_MUN,SEXO,ANO,POB_TOTAL,POB_00_04,POB_05_09,POB_010_014,...,POB_50_54,POB_55_59,POB_60_64,POB_65_69,POB_70_74,POB_75_79,POB_80_84,POB_85_mm,fecha,etiqueta_estado
0,1001,1,Aguascalientes,Aguascalientes,HOMBRES,1990,243891,35865,32966,30713,...,6323,5051,4206,3265,2316,1617,1022,681,1990-01-01,Aguascalientes
1,1001,1,Aguascalientes,Aguascalientes,MUJERES,1990,255948,34812,31927,30190,...,7188,6081,5161,4112,3041,2215,1474,1063,1990-01-01,Aguascalientes
2,1001,1,Aguascalientes,Aguascalientes,HOMBRES,1991,252446,36879,33745,31208,...,6705,5261,4336,3360,2397,1654,1034,715,1991-01-01,Aguascalientes
3,1001,1,Aguascalientes,Aguascalientes,MUJERES,1991,264829,35825,32697,30583,...,7569,6289,5329,4255,3165,2267,1495,1121,1991-01-01,Aguascalientes
4,1001,1,Aguascalientes,Aguascalientes,HOMBRES,1992,261132,37908,34523,31773,...,7108,5486,4461,3451,2491,1694,1042,748,1992-01-01,Aguascalientes


In [10]:
data_pop_m.columns = data_pop_m.columns.str.lower()
data_pop_m.rename(columns={'ano': 'year', 'clave_ent': 'state_id', 'nom_ent': 'state', 'clave': 'muni_id', 'nom_mun': 'municipality', 'sexo': 'sex', 'pob_total': 'pop_total'}, 
                  inplace=True)
data_pop_m['state_id'] = data_pop_m['state_id'].astype('int64')
data_pop_m.columns

Index(['muni_id', 'state_id', 'state', 'municipality', 'sex', 'year',
       'pop_total', 'pob_00_04', 'pob_05_09', 'pob_010_014', 'pob_015_019',
       'pob_20_24', 'pob_25_29', 'pob_30_34', 'pob_35_39', 'pob_40_44',
       'pob_45_49', 'pob_50_54', 'pob_55_59', 'pob_60_64', 'pob_65_69',
       'pob_70_74', 'pob_75_79', 'pob_80_84', 'pob_85_mm', 'fecha',
       'etiqueta_estado'],
      dtype='object')

In [11]:
# Filter data set to 2025 only
pop_filtered_m = data_pop_m[data_pop_m.year == 2025].reset_index(drop=True)

# Filter data set to 2025 only
pop_filtered_m = pop_filtered_m[pop_filtered_m.state_id.isin([12, 16, 15])].reset_index(drop=True)
print(pop_filtered_m.state.unique())

print(pop_filtered_m.shape)
pop_filtered_m.head()

['Guerrero' 'México' 'Michoacán de Ocampo']
(646, 27)


,muni_id,state_id,state,municipality,sex,year,pop_total,pob_00_04,pob_05_09,pob_010_014,...,pob_50_54,pob_55_59,pob_60_64,pob_65_69,pob_70_74,pob_75_79,pob_80_84,pob_85_mm,fecha,etiqueta_estado
0,12001,12,Guerrero,Acapulco de Juárez,HOMBRES,2025,374985,29004,31618,31862,...,21433,19706,17420,13456,9239,5893,3365,2817,2025-01-01,Guerrero
1,12001,12,Guerrero,Acapulco de Juárez,MUJERES,2025,412991,27706,30097,30352,...,26676,23821,20436,16073,11165,7496,4629,3984,2025-01-01,Guerrero
2,12002,12,Guerrero,Ahuacuotzingo,HOMBRES,2025,11745,1511,1480,1410,...,461,400,351,285,228,196,137,134,2025-01-01,Guerrero
3,12002,12,Guerrero,Ahuacuotzingo,MUJERES,2025,13285,1431,1461,1421,...,533,477,406,333,273,243,180,162,2025-01-01,Guerrero
4,12003,12,Guerrero,Ajuchitlán del Progreso,HOMBRES,2025,18054,1918,1914,1743,...,723,671,648,576,455,349,279,246,2025-01-01,Guerrero


In [12]:
df_pop_yearly_m = pop_filtered_m.groupby(['municipality', 'year', 'muni_id', 'state_id', 'state'], as_index=False)['pop_total'].sum()
df_pop_yearly_m

,municipality,year,muni_id,state_id,state,pop_total
0,Acambay de Ruíz Castañeda,2025,15001,15,México,70196
1,Acapulco de Juárez,2025,12001,12,Guerrero,787976
2,Acatepec,2025,12076,12,Guerrero,43859
3,Acolman,2025,15002,15,México,189927
4,Acuitzio,2025,16001,16,Michoacán de Ocampo,11525
...,...,...,...,...,...,...
318,Zitácuaro,2025,16112,16,Michoacán de Ocampo,154483
319,Zumpahuacán,2025,15119,15,México,19879
320,Zumpango,2025,15120,15,México,319152
321,Álvaro Obregón,2025,16003,16,Michoacán de Ocampo,25302


### Protection panel

In [15]:
pol_mun_rates = df_pol_mun.merge(df_pop_yearly_m.loc[:, ['year', 'municipality', 'pop_total', 'muni_id', 'state_id', 'state']], on=['muni_id'], how='inner')
print(pol_mun_rates.shape)
pol_mun_rates.head(10)

(323, 7)


,muni_id,count_pol_mun,year,municipality,pop_total,state_id,state
0,12001,3.0,2025,Acapulco de Juárez,787976,12,Guerrero
1,12002,31.0,2025,Ahuacuotzingo,25030,12,Guerrero
2,12003,7.0,2025,Ajuchitlán del Progreso,37519,12,Guerrero
3,12004,44.0,2025,Alcozauca de Guerrero,22215,12,Guerrero
4,12005,17.0,2025,Alpoyeca,8194,12,Guerrero
5,12006,0.0,2025,Apaxtla,11172,12,Guerrero
6,12007,33.0,2025,Arcelia,35697,12,Guerrero
7,12008,20.0,2025,Atenango del Río,9384,12,Guerrero
8,12009,15.0,2025,Atlamajalcingo del Monte,6249,12,Guerrero
9,12010,39.0,2025,Atlixtac,29380,12,Guerrero


In [16]:
pol_mun_rates['pol_mun_rate_100k'] = pol_mun_rates['count_pol_mun'] / pol_mun_rates['pop_total'] * 100000
pol_mun_rates.head()

,muni_id,count_pol_mun,year,municipality,pop_total,state_id,state,pol_mun_rate_100k
0,12001,3.0,2025,Acapulco de Juárez,787976,12,Guerrero,0.380722
1,12002,31.0,2025,Ahuacuotzingo,25030,12,Guerrero,123.851378
2,12003,7.0,2025,Ajuchitlán del Progreso,37519,12,Guerrero,18.657214
3,12004,44.0,2025,Alcozauca de Guerrero,22215,12,Guerrero,198.064371
4,12005,17.0,2025,Alpoyeca,8194,12,Guerrero,207.468880


In [17]:
# Merges the police rates per municipality with the presence of federal law enforcement
df_prot = (
    pol_mun_rates[["muni_id", "municipality", "year", "state_id", "state", "pol_mun_rate_100k"]]
    .merge(df_fed_mun[["muni_id", "federal_presence"]], on="muni_id", how="left")
)
df_prot["federal_presence"] = df_prot["federal_presence"].fillna(0)

print(df_prot.shape)
df_prot.head()

(323, 7)


,muni_id,municipality,year,state_id,state,pol_mun_rate_100k,federal_presence
0,12001,Acapulco de Juárez,2025,12,Guerrero,0.380722,1
1,12002,Ahuacuotzingo,2025,12,Guerrero,123.851378,0
2,12003,Ajuchitlán del Progreso,2025,12,Guerrero,18.657214,0
3,12004,Alcozauca de Guerrero,2025,12,Guerrero,198.064371,0
4,12005,Alpoyeca,2025,12,Guerrero,207.468880,0


In [18]:
df_prot.federal_presence.value_counts()

federal_presence
0    302
1     21
Name: count, dtype: int64

In [20]:
# Creates index
df_prot["prot_raw"] = df_prot["pol_mun_rate_100k"] + 50 * df_prot["federal_presence"]

df_prot["prot_idx"] = (
    (df_prot["prot_raw"] - df_prot["prot_raw"].min()) /
    (df_prot["prot_raw"].max() - df_prot["prot_raw"].min())
)

In [21]:
df_prot['year'] = df_prot['year'].astype('int64')
df_prot['state_id'] = df_prot['state_id'].astype('int64')
df_prot['state'] = df_prot['state'].astype('str')
df_prot['muni_id'] = df_prot['muni_id'].astype('str')
df_prot['municipality'] = df_prot['municipality'].astype('str')

df_prot['pol_mun_rate_100k'] = df_prot['pol_mun_rate_100k'].round(1).astype(float)
df_prot['federal_presence'] = df_prot['federal_presence'].astype('int64')

In [22]:
print(df_prot.shape)
df_prot.head(10)

(323, 9)


,muni_id,municipality,year,state_id,state,pol_mun_rate_100k,federal_presence,prot_raw,prot_idx
0,12001,Acapulco de Juárez,2025,12,Guerrero,0.4,1,50.380722,0.104702
1,12002,Ahuacuotzingo,2025,12,Guerrero,123.9,0,123.851378,0.257391
2,12003,Ajuchitlán del Progreso,2025,12,Guerrero,18.7,0,18.657214,0.038774
3,12004,Alcozauca de Guerrero,2025,12,Guerrero,198.1,0,198.064371,0.411622
4,12005,Alpoyeca,2025,12,Guerrero,207.5,0,207.468880,0.431166
5,12006,Apaxtla,2025,12,Guerrero,0.0,0,0.000000,0.000000
6,12007,Arcelia,2025,12,Guerrero,92.4,0,92.444743,0.192121
7,12008,Atenango del Río,2025,12,Guerrero,213.1,0,213.128730,0.442929
8,12009,Atlamajalcingo del Monte,2025,12,Guerrero,240.0,0,240.038406,0.498853
9,12010,Atlixtac,2025,12,Guerrero,132.7,0,132.743363,0.275870


#### Save data

In [24]:
df_prot.to_csv(out_protection_panel, index=False)